# Building an Intelligent RAG-based Chatbot using LangChain and LLMs

### PDF Question Answering System using Retrieval-Augmented Generation

## Project Description

This project implements a simple Retrieval-Augmented Generation (RAG) based
chatbot that allows users to ask questions from an uploaded PDF document.

The system extracts text from the PDF and divides the text into smaller chunks.
These chunks are converted into embeddings using a Hugging Face embedding model
and stored in ChromaDB.

When the user asks a question, the system searches ChromaDB to find the most
relevant document chunks. The retrieved information is then provided to a Large
Language Model through the Groq API.

The LLM uses the retrieved information to generate an answer based on the
uploaded document.

The entire project is implemented in a single Google Colab notebook.

## Project Objective

The main objective of this project is to develop a simple RAG-based chatbot
that can answer questions based on the content of an uploaded PDF document.

The objectives are:

1. Upload and extract text from a PDF document.
2. Split the extracted text into smaller chunks.
3. Convert the text chunks into embeddings.
4. Store the embeddings in a vector database.
5. Retrieve relevant text based on the user's question.
6. Use an LLM to generate an answer using the retrieved information.
7. Provide a simple interface for users to ask questions.

## System Architecture

The RAG chatbot works using the following steps:

PDF Document
↓
Text Extraction
↓
Text Chunking
↓
Embedding Generation
↓
ChromaDB Vector Store
↓
User Question
↓
Relevant Document Retrieval
↓
Prompt with Retrieved Context
↓
Large Language Model
↓
Final Answer

### Architecture Explanation

First, the PDF document is uploaded and its text is extracted.

The extracted text is divided into smaller chunks so that the information can
be processed and retrieved efficiently.

Each text chunk is converted into a numerical representation called an
embedding. These embeddings are stored in ChromaDB.

When the user asks a question, the question is compared with the stored
embeddings to retrieve the most relevant text chunks.

The retrieved text is added as context to the user's question and sent to the
Large Language Model through the Groq API.

Finally, the LLM generates an answer based on the retrieved PDF information.

## Install Required Libraries

The following libraries are required to build the RAG chatbot.

- LangChain is used to build and manage the RAG pipeline.
- PyPDF is used to extract text from PDF documents.
- Hugging Face Sentence Transformers are used to generate embeddings.
- ChromaDB is used to store and retrieve document embeddings.
- Groq is used to access the Large Language Model.

In [1]:
!pip -q install langchain langchain-community langchain-groq langchain-huggingface langchain-chroma chromadb pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206

## Import Required Libraries

The required Python libraries are imported for PDF processing, text splitting,
embedding generation, vector storage, LLM integration, and the chatbot interface.

In [2]:
import langchain
import chromadb
import pypdf
import sentence_transformers

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("LangChain:", langchain.__version__)
print("ChromaDB:", chromadb.__version__)
print("PyPDF:", pypdf.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

print("\n✅ RAG libraries imported successfully!")

LangChain: 1.4.0
ChromaDB: 1.5.9
PyPDF: 6.19.0
Sentence Transformers: 5.7.0

✅ RAG libraries imported successfully!


## Configure Groq API and Large Language Model

Groq is used to access the Large Language Model (LLM) for generating responses.

The API key is stored securely using Google Colab Secrets instead of writing
the key directly inside the notebook.

The LLM receives the user's question together with relevant information
retrieved from the PDF and generates the final response.

In [3]:
from google.colab import userdata

groq_api_key=userdata.get("GROQ_API_KEY")
if groq_api_key:
  print("✅ GROQ API key loaded successfully!")
else:
  print("❌ GROQ API key not found!")

✅ GROQ API key loaded successfully!


### Initialize the Large Language Model

The Groq API is connected to LangChain using ChatGroq.
The temperature is set to 0 to produce more consistent responses.

In [20]:
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="openai/gpt-oss-120b",
    temperature=0
)

print("LLM initialized successfully!")

LLM initialized successfully!


## Upload PDF Document

The user uploads a PDF document from their computer.

The uploaded PDF will be used as the knowledge source for the RAG chatbot.

In [5]:
from google.colab import files

uploaded = files.upload()

Saving Sample-pdf.pdf to Sample-pdf.pdf


In [6]:
pdf_filename = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_filename)

Uploaded PDF: Sample-pdf.pdf


## Extract Text from PDF

PyPDFLoader is used to read the uploaded PDF and extract its text.

Each PDF page is converted into a LangChain Document object. The object contains
the extracted text and metadata such as the page number and source document.

In [7]:
from langchain_community.document_loaders import PyPDFLoader

# Load the uploaded PDF
loader = PyPDFLoader(pdf_filename)

# Extract the PDF content
documents = loader.load()

print("PDF loaded successfully!")
print("Number of pages:", len(documents))

/tmp/ipykernel_646/1577334227.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF loaded successfully!
Number of pages: 10


In [8]:
print("Content from the first page:\n")
print(documents[0].page_content[:1000])

Content from the first page:

Sample PDF Document
Robert Maron
Grzegorz Grudzi´nski
February 20, 1999


## Text Chunking

The extracted PDF text is divided into smaller chunks using
RecursiveCharacterTextSplitter.

Smaller chunks make it easier for the system to retrieve only the information
that is relevant to a user's question.

A small overlap is maintained between consecutive chunks so that important
context is not lost at chunk boundaries.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of original pages:", len(documents))
print("Number of text chunks:", len(chunks))

Number of original pages: 10
Number of text chunks: 12


In [10]:
print("First text chunk:\n")
print(chunks[0].page_content)

print("\nMetadata:")
print(chunks[0].metadata)

First text chunk:

Sample PDF Document
Robert Maron
Grzegorz Grudzi´nski
February 20, 1999

Metadata:
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20200629161449', 'source': 'Sample-pdf.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1'}


## Generate Text Embeddings

An embedding is a numerical representation of text.

The Hugging Face model `all-MiniLM-L6-v2` is used to convert document chunks
into vectors. Text with similar meanings will have similar vector
representations.

These embeddings allow the system to perform semantic search instead of
depending only on exact keyword matching.

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [12]:
test_embedding = embedding_model.embed_query(
    "What is artificial intelligence?"
)

print("Embedding created successfully!")
print("Vector length:", len(test_embedding))
print("First 10 values:")
print(test_embedding[:10])

Embedding created successfully!
Vector length: 384
First 10 values:
[-0.015781095251441002, 0.014908617362380028, 0.00909555796533823, 0.03131742775440216, -0.02170826494693756, -0.04852483794093132, 0.04887937009334564, 0.028673212975263596, -0.02559719979763031, 0.05404298007488251]


## Create ChromaDB Vector Store

ChromaDB is used as the vector database for the project.

The document chunks are converted into embeddings and stored in ChromaDB.
This allows the system to search for document chunks that are semantically
similar to a user's question.

In [13]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("ChromaDB vector store created successfully!")
print("Number of stored chunks:", vector_store._collection.count())

ChromaDB vector store created successfully!
Number of stored chunks: 12


## Test Semantic Retrieval

The following code tests the retrieval system.

The user's question is compared with the stored document embeddings, and
ChromaDB returns the three most relevant document chunks.

In [14]:
query = "What is this document about?"

results = vector_store.similarity_search(
    query,
    k=3
)

print("Top relevant chunks:\n")

for i, doc in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print(doc.page_content[:500])
    print("Page:", doc.metadata.get("page", "Unknown"))
    print()

Top relevant chunks:

--- Result 1 ---
Sample PDF Document
Robert Maron
Grzegorz Grudzi´nski
February 20, 1999
Page: 0

--- Result 2 ---
10 CHAPTER 1. TEMPLATE
– large
– Large
– LARGE
– huge
– Huge
Page: 9

--- Result 3 ---
4 CONTENTS
Page: 3



## RAG Question-Answering Pipeline

The following function implements the main Retrieval-Augmented Generation
pipeline.

When a user asks a question:

1. Relevant document chunks are retrieved from ChromaDB.
2. The retrieved chunks are combined to create context.
3. The context and user's question are added to a prompt.
4. The prompt is sent to the Large Language Model.
5. The LLM generates an answer based on the retrieved document information.

In [15]:
def ask_rag(question):
    # Step 1: Retrieve relevant chunks from ChromaDB
    relevant_docs = vector_store.similarity_search(
        question,
        k=3
    )

    # Step 2: Combine retrieved chunks into context
    context = "\n\n".join(
        doc.page_content for doc in relevant_docs
    )

    # Step 3: Create an augmented prompt
    prompt = f"""
You are a helpful PDF question-answering assistant.

Answer the question using ONLY the information provided in the context below.

If the answer cannot be found in the context, say:
"I could not find this information in the uploaded document."

Context:
{context}

Question:
{question}

Answer:
"""

    # Step 4: Send the augmented prompt to the LLM
    response = llm.invoke(prompt)

    # Step 5: Return answer and retrieved documents
    return response.content, relevant_docs


print("RAG pipeline created successfully!")

RAG pipeline created successfully!


In [16]:
question = "What is this document about?"

answer, sources = ask_rag(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

print("\nSources:")

for i, doc in enumerate(sources, start=1):
    page = doc.metadata.get("page", "Unknown")

    if isinstance(page, int):
        page = page + 1

    print(f"Source {i}: {pdf_filename}, Page {page}")

Question:
What is this document about?

Answer:
The document appears to be about a template.

Sources:
Source 1: Sample-pdf.pdf, Page 1
Source 2: Sample-pdf.pdf, Page 10
Source 3: Sample-pdf.pdf, Page 4


In [17]:
question = "Who is the current Prime Minister of Japan?"

answer, sources = ask_rag(question)

print("Question:", question)
print("\nAnswer:", answer)

Question: Who is the current Prime Minister of Japan?

Answer: I could not find this information in the uploaded document.


## Simple Chatbot Interface

A simple chatbot interface is created using `ipywidgets`.

The user can enter a question about the uploaded PDF and click the Ask button.
The question is passed to the RAG pipeline, which retrieves relevant information
and generates an answer.

The source page numbers are also displayed with the response.

In [19]:
import ipywidgets as widgets
from IPython.display import display

question_box = widgets.Text(
    placeholder="Enter your question here"
)

ask_button = widgets.Button(
    description="Ask"
)

output = widgets.Output()

def ask_question(button):
    question = question_box.value

    if question:
        answer, sources = ask_rag(question)

        with output:
            output.clear_output()

            print("Question:", question)
            print("\nAnswer:", answer)

            print("\nSources:")
            for doc in sources:
                page = doc.metadata.get("page", "Unknown")

                if isinstance(page, int):
                    page += 1

                print(f"{pdf_filename} - Page {page}")

ask_button.on_click(ask_question)

display(question_box)
display(ask_button)
display(output)

Text(value='', placeholder='Enter your question here')

Button(description='Ask', style=ButtonStyle())

Output()

## Testing

The RAG chatbot was tested using both document-related and unrelated questions.

### Test 1: Document-Related Question

**Question:** What is this document about?

**Result:** The chatbot retrieved relevant chunks from the PDF and generated
an answer based on the document content.

### Test 2: Question Not Available in the Document

**Question:** Who is the current Prime Minister of Japan?

**Result:** The chatbot responded:

"I could not find this information in the uploaded document."

This test shows that the chatbot attempts to restrict its answers to information
retrieved from the uploaded document.

## Results

The developed RAG chatbot successfully processes PDF documents and allows users
to ask questions about their content.

The system was able to:

- Extract text from an uploaded PDF.
- Split the document into smaller text chunks.
- Generate embeddings for semantic search.
- Store and retrieve document information using ChromaDB.
- Retrieve relevant document chunks for user questions.
- Generate answers using the retrieved context and an LLM.
- Display source page information with the generated response.
- Avoid answering questions when the required information was not available
  in the retrieved document context.

The testing shows that the basic RAG pipeline works successfully for
document-based question answering.

## Limitations

The current system has the following limitations:

1. The chatbot currently processes one PDF document at a time.
2. The quality of the answer depends on the quality of the extracted PDF text.
3. Scanned PDFs may require OCR before their text can be processed.
4. The chatbot requires an internet connection to access the Groq API.
5. The quality of retrieval depends on the chunk size and embedding model.
6. The system retrieves a fixed number of document chunks for each question.
7. The current chatbot interface is basic and designed mainly for demonstration.

## Conclusion

This project demonstrates the implementation of a simple
Retrieval-Augmented Generation (RAG) based PDF chatbot.

The system combines document processing, text chunking, embeddings,
vector-based retrieval, and a Large Language Model to answer questions
using information from an uploaded PDF.

ChromaDB is used to retrieve relevant document chunks, while the Groq API
provides access to the Large Language Model used for response generation.

The project demonstrates how external document knowledge can be combined
with an LLM to create a simple document question-answering system.

The entire project was implemented in a single Google Colab notebook,
making it easy to execute, understand, and demonstrate.